# RAG Workflow — LlamaIndex + Ollama

End-to-end Retrieval-Augmented Generation pipeline using:
- **LlamaIndex** for document loading, chunking, indexing, and querying
- **Ollama** for a local LLM (`llama3.2`) and embedding model (`nomic-embed-text`)

**Prerequisites** (run once in your terminal before opening this notebook):
```bash
# 1. Start Ollama and pull models
ollama serve
ollama pull llama3.2
ollama pull nomic-embed-text

# 2. Sync the uv project environment (installs all dependencies from uv.lock)
uv sync

# 3. Launch the notebook inside the uv-managed environment
uv run jupyter lab rag_workflow.ipynb
```

> **Note:** `uv sync` reads `pyproject.toml` + `uv.lock` and creates/updates `.venv` automatically.  
> You never need to manually activate the virtualenv — `uv run` handles it.

---
## Section 1 — Setup & Imports

Run the optional install cell once if packages are not yet installed, then import everything needed for the pipeline.

In [ ]:
# Optional: add a new dependency without leaving the notebook.
# Uncomment and run to add a package to pyproject.toml + uv.lock and install it.
# import subprocess
# subprocess.run(["uv", "add", "some-package"], check=True)

In [ ]:
import os
import sys
from pathlib import Path

# Make src/ modules importable
sys.path.insert(0, str(Path().resolve() / "src"))

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    Settings,
    load_index_from_storage,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

print("Imports OK")

---
## Section 2 — Configuration

Edit this single cell to customise the entire pipeline.

In [ ]:
# ── Model endpoints ──────────────────────────────────────────────────────────
OLLAMA_BASE_URL = "http://localhost:11434"
LLM_MODEL       = "llama3.2-4k"       # custom model with num_ctx=4096 baked in
EMBED_MODEL     = "nomic-embed-text"   # fallback: "llama3.2"

# ── Context window ────────────────────────────────────────────────────────────
# Kept for reference — the actual limit is enforced by the llama3.2-4k Modelfile.
# Create it once with: printf 'FROM llama3.2\nPARAMETER num_ctx 4096\n' | ollama create llama3.2-4k
NUM_CTX         = 4096

# ── Chunking ─────────────────────────────────────────────────────────────────
CHUNK_SIZE      = 512
CHUNK_OVERLAP   = 50

# ── Retrieval ─────────────────────────────────────────────────────────────────
TOP_K           = 3

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR        = "./data"
PERSIST_DIR     = "./storage"

print(f"LLM : {LLM_MODEL}  |  Embed : {EMBED_MODEL}  |  ctx : {NUM_CTX}")
print(f"Chunk size : {CHUNK_SIZE}  |  Overlap : {CHUNK_OVERLAP}  |  Top-K : {TOP_K}")
print(f"Data dir   : {DATA_DIR}  |  Persist dir : {PERSIST_DIR}")

---
## Section 3 — LLM & Embedding Initialisation

Instantiate both models, register them on the `Settings` singleton so all downstream components inherit them, then run connectivity checks.

In [ ]:
# Instantiate LLM
llm = Ollama(
    model=LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    request_timeout=120.0,
    temperature=0.1,
    context_window=NUM_CTX,
)

# Instantiate embedding model
embed_model = OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

# Register on the global Settings singleton
Settings.llm = llm
Settings.embed_model = embed_model

print("LLM and embedding model registered on Settings.")

In [ ]:
# ── LLM connectivity check ────────────────────────────────────────────────────
try:
    test_response = llm.complete("Reply with exactly: OK")
    print(f"LLM connection OK  →  {str(test_response).strip()}")
except Exception as e:
    print(f"LLM connection FAILED: {e}")
    print("Make sure `ollama serve` is running and `llama3.2` is pulled.")

In [ ]:
# ── Embedding connectivity check ──────────────────────────────────────────────
try:
    test_embedding = embed_model.get_text_embedding("connectivity test")
    print(f"Embedding connection OK  →  vector dim = {len(test_embedding)}")
except Exception as e:
    print(f"Embedding connection FAILED: {e}")
    print(f"Make sure `ollama pull {EMBED_MODEL}` has been run.")

---
## Section 4 — Document Loading

`SimpleDirectoryReader` recursively loads `.txt`, `.md`, and `.pdf` files from `DATA_DIR`. The `required_exts` filter prevents it from accidentally ingesting `.ipynb` or hidden files.

In [ ]:
reader = SimpleDirectoryReader(
    input_dir=DATA_DIR,
    recursive=True,
    required_exts=[".txt", ".md", ".pdf"],
)

documents = reader.load_data()

print(f"Loaded {len(documents)} document(s)\n")
for doc in documents:
    filename = doc.metadata.get("file_name", "unknown")
    print(f"  • {filename:40s}  {len(doc.text):>6,} chars")

---
## Section 5 — Index Creation with Persistence

On the **first run** the index is built from documents and saved to `PERSIST_DIR`.  
On **subsequent runs** the index is loaded from disk — no re-embedding required.

In [ ]:
if os.path.exists(PERSIST_DIR) and os.listdir(PERSIST_DIR):
    # ── Load existing index from disk ─────────────────────────────────────────
    print(f"Found existing index at '{PERSIST_DIR}'. Loading from disk...")
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)
    print("Index loaded successfully.")
else:
    # ── Build new index from documents ────────────────────────────────────────
    print("No existing index found. Building from documents (this may take a moment)...")

    splitter = SentenceSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

    index = VectorStoreIndex.from_documents(
        documents,
        transformations=[splitter],
        show_progress=True,
    )

    os.makedirs(PERSIST_DIR, exist_ok=True)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    print(f"Index built and saved to '{PERSIST_DIR}'.")

---
## Section 6 — Query Engine Setup

Wrap the index in a query engine. `similarity_top_k` controls how many chunks are retrieved; `response_mode="compact"` fits as many chunks as possible into a single prompt.

In [ ]:
query_engine = index.as_query_engine(
    similarity_top_k=TOP_K,
    response_mode="compact",
)

print(f"Query engine ready  |  top_k={TOP_K}  |  response_mode=compact")

---
## Section 7 — Interactive Q&A Demo

### 7a. Single Query
Edit `QUESTION` and re-run the cell.

In [ ]:
QUESTION = "What is Retrieval-Augmented Generation and what are its main benefits?"

response = query_engine.query(QUESTION)

print(f"Q: {QUESTION}")
print("\nA:", response)

### 7b. Source Attribution

Inspect which document chunks were retrieved, their similarity scores, and an excerpt of their content.

In [ ]:
print(f"Retrieved {len(response.source_nodes)} source node(s):\n")

for i, node in enumerate(response.source_nodes, start=1):
    filename = node.metadata.get("file_name", "unknown")
    score    = node.score if node.score is not None else float("nan")
    excerpt  = node.get_content()[:200].replace("\n", " ")
    print(f"[{i}] File  : {filename}")
    print(f"    Score : {score:.4f}")
    print(f"    Text  : {excerpt}...")
    print()

### 7c. Multi-Question Loop

In [ ]:
demo_questions = [
    "What is LlamaIndex and what are its key components?",
    "How does Ollama work and which models does it support?",
    "What is the difference between the indexing phase and the querying phase in RAG?",
    "When should you use RAG instead of fine-tuning a model?",
]

for q in demo_questions:
    r = query_engine.query(q)
    answer = str(r).strip()
    print(f"Q: {q}")
    print(f"A: {answer[:400]}{'...' if len(answer) > 400 else ''}")
    print("-" * 70)

---
## Section 8 — (Optional) Streaming

Enable token-by-token streaming for a more interactive experience. Useful for longer answers.

In [ ]:
streaming_engine = index.as_query_engine(
    similarity_top_k=TOP_K,
    response_mode="compact",
    streaming=True,
)

STREAM_QUESTION = "Explain the chunking step in RAG and why chunk overlap matters."

print(f"Q: {STREAM_QUESTION}\n")
streaming_response = streaming_engine.query(STREAM_QUESTION)

for token in streaming_response.response_gen:
    print(token, end="", flush=True)
print()

---
## Section 9 — Multi-Round Chat Engine

Unlike the query engine in Sections 6–7, the **chat engine** maintains conversation history across turns. It uses `condense_plus_context` mode:

1. **Condense** — rewrites the latest question in light of prior exchanges into a standalone question
2. **Retrieve** — runs similarity search with the condensed question
3. **Answer** — passes retrieved chunks + full chat history to the LLM

This means follow-up questions like *"What did you mean by that?"* or *"Give me more detail on point 2"* work correctly.

In [ ]:
chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context",
    similarity_top_k=TOP_K,
    verbose=False,
)

print("Chat engine ready.")

### 9a. Multi-Round Demo

Run this cell to see how context carries across turns — the second and third questions are deliberate follow-ups that rely on the prior answer.

In [ ]:
chat_engine.reset()  # start a fresh conversation

exchanges = [
    "What do these documents reveal about the investigations?",
    "Who are the key individuals mentioned?",
    "Can you elaborate on the third point from your last answer?",  # tests memory
]

for question in exchanges:
    print(f"You : {question}")
    response = chat_engine.chat(question)
    print(f"AI  : {str(response).strip()}\n")
    print("-" * 70)

### 9b. Interactive Single-Turn Cell

Edit `CHAT_QUESTION` and re-run this cell repeatedly. Each call adds to the conversation history — the engine remembers all prior turns.  
Run the cell above (`chat_engine.reset()`) to start a new conversation.

In [ ]:
CHAT_QUESTION = "What is the significance of these disclosures?"

response = chat_engine.chat(CHAT_QUESTION)
print(f"You : {CHAT_QUESTION}")
print(f"AI  : {str(response).strip()}")

# Source nodes for this turn
if hasattr(response, "source_nodes") and response.source_nodes:
    print(f"\nSources ({len(response.source_nodes)}):")
    for node in response.source_nodes:
        filename = node.metadata.get("file_name", "unknown")
        score = node.score if node.score is not None else float("nan")
        print(f"  [{score:.3f}] {filename}")